# Synthetic vs. Real — Google Colab GPU training

This notebook compares eight YOLOv8n models with **300 training images each**
(five completed models are reused and three new mixtures are trained):
real only, AI only, rendered only, 50% AI + 25% rendered + 25% real,
50% real + 25% AI + 25% rendered, 50% AI + 50% real,
75% real + 25% AI, and 75% AI + 25% real. All use the same **7 real validation images
and 22 real test images**. Classes: `0 worker`, `1 helmet`, `2 vest`.

**Existing labels are used without further annotation review**, as requested.
Previously rejected images and missing files are excluded. Some selected
training labels remain unresolved in the old review ledger; structural checks
do not certify their accuracy. The full original v4 review remains incomplete.
Holdouts are small and contain correlated frames, so results are preliminary.
Do not compare these scores directly with the bundled v2 model scores.

**Before running:**
1. Keep your original dataset ZIP and completed run folder in Drive. Upload only the small `synthreal_experiment_update.zip` into a `SynthReal` folder in **My Drive**.
2. In Colab choose **Runtime → Change runtime type → GPU** (T4 if available).
3. Run the cells in order. Allow the Google Drive connection when prompted.

Your computer only needs a browser and internet. Training runs on Colab's GPU.
Colab GPU availability and runtime limits vary; checkpoints are saved to Drive.
[Colab FAQ](https://research.google.com/colaboratory/faq.html) ·
[Ultralytics training and resume documentation](https://docs.ultralytics.com/modes/train/)


## 1. Check the GPU


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, reconnect, then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)


## 2. Connect Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Reuse your existing dataset and completed models
Upload only `synthreal_experiment_update.zip` to `MyDrive/SynthReal/`.
This cell uses the dataset already in this runtime, or reads the original
`synthreal_300.zip` already in Drive after a runtime reset. It copies the five
completed `best.pt` checkpoints from your previous Drive run and verifies hashes.
No images or models need to be uploaded again. If you moved the original ZIP or
run folder, change `OLD_DATA_ZIP` or `PREVIOUS_RUN` below. Original files stay intact.


In [ ]:
from pathlib import Path
import hashlib
import shutil
import stat
import sys
import tempfile
import zipfile

DRIVE = Path('/content/drive/MyDrive/SynthReal')
UPDATE_ZIP = DRIVE / 'synthreal_experiment_update.zip'
EXISTING_DATA = Path('/content/synthreal_5ea8644bd890')
OLD_DATA_ZIP = DRIVE / 'synthreal_300.zip'
PREVIOUS_RUN = DRIVE / 'runs/existing_labels_300_seed42_5ea8644b'
EXPECTED_SHA256 = 'bcf6516642370c6182efc4ddabe5b5e3250ddf64e9562656c4be7e9d6c588f12'
DATA_ROOT = Path('/content/synthreal_update_' + EXPECTED_SHA256[:12])

if not UPDATE_ZIP.is_file():
    raise FileNotFoundError(f'Upload only the small update ZIP to {UPDATE_ZIP}')
if hashlib.sha256(UPDATE_ZIP.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise ValueError('Update ZIP and notebook do not match; use the latest pair.')
if not DATA_ROOT.exists():
    temporary = Path(tempfile.mkdtemp(prefix='synthreal_update_', dir='/content'))
    with zipfile.ZipFile(UPDATE_ZIP) as bundle:
        for member in bundle.infolist():
            p = Path(member.filename)
            if p.is_absolute() or '..' in p.parts or chr(92) in member.filename or ':' in member.filename or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError('Unsafe update member: ' + member.filename)
        bundle.extractall(temporary)
    temporary.replace(DATA_ROOT)
sys.path.insert(0, str(DATA_ROOT))
# Avoid reusing a helper imported by an older notebook in the same session.
for module in ('colab_training', 'colab_existing_data'):
    sys.modules.pop(module, None)
from colab_existing_data import prepare_existing_data
prepare_existing_data(DATA_ROOT, EXISTING_DATA, OLD_DATA_ZIP, PREVIOUS_RUN)
print('Reused existing dataset and all five completed models:', DATA_ROOT)


## 4. Install the training library


In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', str(DATA_ROOT / 'requirements-colab.txt')])
# Keep the CUDA-enabled torch installation supplied by Colab.
sys.path.insert(0, str(DATA_ROOT))
from colab_training import verify_and_prepare, run_training


## 5. Validate every selected image, label, split, and checksum
This step creates paths for the current Colab machine. It rejects changed
annotations, exact duplicate images, overlapping holdouts, and known source
scene groups crossing splits. It does not certify that every original source
scene is independent; source video IDs are unavailable.


In [ ]:
prepared = verify_and_prepare(DATA_ROOT)
if prepared['manifest']['train_images_per_experiment'] != 300:
    raise ValueError('This notebook requires the 300-image comparison bundle.')
import pandas as pd
table = []
for experiment in prepared['experiments']:
    table.append({'experiment': experiment['name'],
                  **experiment['sources'],
                  'train': len(experiment['splits']['train']),
                  'val': len(experiment['splits']['val']),
                  'test': len(experiment['splits']['test'])})
display(pd.DataFrame(table).fillna(0))
print('All selected files and split checks passed. Annotation accuracy has not been certified.')


## 6. Train or resume the three new mixtures
The defaults use 50 epochs, 640-pixel input, batch 8, seed 42, AdamW, and GPU 0.
The five completed models are included in this bundle. Their source
membership, image/label hashes, settings, initialization, and checkpoint
checksums must match before reuse. Their saved test scores are retained.
Only the three new AI/real mixtures train by default; each starts from the
same YOLOv8n pretrained weights. The first
training call downloads those weights. Model selection uses validation only;
the test set is evaluated after training.

After a disconnect, reconnect to a GPU and rerun the notebook. Completed
experiments are skipped and incomplete ones resume from `weights/last.pt`.
At least one saved epoch is needed to resume. Periodic epoch checkpoints are
also kept. Do not change data or settings while resuming the same run folder.
If you deliberately change settings, give `RUN_NAME` a new value and set
`REUSE_BASELINES = False` to retrain all eight with matching settings. If GPU memory
is insufficient, use a smaller batch in a **new** run folder for all eight models.


In [ ]:
EPOCHS = 50
BATCH = 8
RUN_NAME = 'expanded_8_models_300_seed42_' + EXPECTED_SHA256[:8]
OUTPUT = Path('/content/drive/MyDrive/SynthReal/runs') / RUN_NAME
REUSE_BASELINES = True  # Set False in a NEW run folder to retrain all eight.
results = run_training(prepared, OUTPUT, epochs=EPOCHS, batch=BATCH,
                       reuse_baselines=REUSE_BASELINES)
print('Saved results and checkpoints:', OUTPUT)


## 7. Read the comparison


In [ ]:
comparison = pd.read_csv(OUTPUT / 'comparison_metrics.csv')
display(comparison)
print('All models have best.pt and test_metrics.json; newly trained models also have last.pt and plots.')
print('Small, curated holdouts: treat these metrics as preliminary.')


## 8. Export the completed models and metrics
The ZIP stays in Google Drive. It includes all eight best checkpoints, the
dataset manifest, settings, and measured metrics. Full resumable runs remain
in the run folder; this smaller export omits optimizer checkpoints.


In [ ]:
export_path = OUTPUT.parent / (RUN_NAME + '_models_and_metrics.zip')
paths = [OUTPUT / name for name in ['comparison_metrics.csv', 'run_specification.json',
         'dataset_manifest.json', 'environment.json', 'initial_weights.json', 'status.json']]
for experiment in prepared['experiments']:
    run = OUTPUT / experiment['name']
    paths.extend([run / 'weights/best.pt', run / 'test_metrics.json'])
if (OUTPUT / 'reused_baselines.json').is_file():
    paths.append(OUTPUT / 'reused_baselines.json')
missing = [str(path) for path in paths if not path.is_file()]
if missing:
    raise RuntimeError('Complete training before export. Missing: ' + ', '.join(missing))
temporary_export = export_path.with_suffix('.tmp')
with zipfile.ZipFile(temporary_export, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in paths:
        bundle.write(path, path.relative_to(OUTPUT).as_posix())
temporary_export.replace(export_path)
print('Download this file from Google Drive:', export_path)
